## Module 6-1 Statistical Testing and Regression

### 1. t-tests

In [ ]:
from scipy import stats
import numpy as np

#### 1.1. One-sample t-test

Used to test whether the mean of a sample differs from a known value (e.g., population mean μ₀).

In [ ]:
# Example data
data = np.array([5.1, 5.3, 4.9, 5.0, 5.2])

# Hypothesized population mean
mu_0 = 5.0

# One-sample t-test
t_stat, p_val = stats.ttest_1samp(data, mu_0)

print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")

#### 1.2. Independent two-sample t-test

Used to compare the means of two independent groups (e.g., control vs. treatment).

In [ ]:
group1 = np.array([4.9, 5.1, 5.0, 4.8, 5.2])
group2 = np.array([5.4, 5.6, 5.5, 5.7, 5.8])

# Equal variance assumed (default)
t_stat, p_val = stats.ttest_ind(group1, group2, equal_var=True)

print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")

#### 1.3. Paired (dependent) t-test

Used when comparing two related samples (e.g., before vs. after measurements on the same subjects).

In [ ]:
before = np.array([10, 12, 13, 12, 11])
after  = np.array([11, 14, 13, 13, 12])

# Paired t-test
t_stat, p_val = stats.ttest_rel(before, after)

print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")

### 2. Linear Regressions

Section 1 asked whether *one* mean differs from another. Almost every empirical AccFin paper asks a harder question: does `y` move with `x`, **holding a list of other things constant**? That is a regression, and in published work it almost never stops at a one-line call to a regression function — the specification also has to decide what the standard errors are clustered on and what fixed effects are absorbed.

This section works one real specification end to end: the baseline model of **Kim, Li, Lu and Yu (2016, *JAE*)**, Section 3.3, estimated on the option volatility smirk panel we built in [3-1e_Option_Volatility_Smirk.ipynb](<../Module 3-Data Collection/3-1e_Option_Volatility_Smirk.ipynb>).

| | |
| --- | --- |
| **2.1** | The empirical model, and which pieces of it we can estimate |
| **2.2** | Merging the remaining CRSP controls (translating `Smirk_FYear.sas`) |
| **2.3** | Building the estimation sample: filters and winsorization |
| **2.4** | A first regression with `pyfixest` |
| **2.5** | Standard errors: clustering by firm *and* year |
| **2.6** | Fixed effects: absorbing firm and year effects |
| **2.7** | Assembling a regression table |
| **2.8** | Exercises |

> **What you need to run this.** Section 2.2 queries WRDS (CRSP only — no OptionMetrics subscription needed) and starts from `data/Smirk_FirmYear.csv`, the output of the Module 3 exercise. Sections 2.3 onward read the saved panel `data/Smirk_FYear.csv`, so if you cannot run the WRDS steps you can still do all of the regression work.

#### 2.1. The empirical model

Kim et al. (2016) test whether financial statement comparability reduces expected crash risk, proxied by the option volatility smirk. Their Eq. (7) is:

$$
\begin{aligned}
IV\_SKEW_{it} = \;& \beta_0 + \beta_1 COMPACCT_{it} + \beta Controls_{it} + \varepsilon_{it}
\end{aligned}
$$

We are not replicating the paper — the variable of interest, $COMPACCT$, needs the De Franco et al. (2011) comparability estimation, which is a project in itself. **The goal here is the mechanics of estimating a specification like this.** So we drop $COMPACCT$ and the controls that require data we have not collected, and keep the eight that our panel can support:

| Paper's variable | Our column | Source | Built in |
| --- | --- | --- | --- |
| $IV\_SKEW$ (dependent) | `iv_skew` | OptionMetrics | 3-1e |
| $ATM\_IV$ | `atm_iv` | OptionMetrics | 3-1e |
| $FIRM\_SIZE$ | `firm_size` | Compustat | 3-1e |
| $LEVERAGE$ | `leverage` | Compustat | 3-1e |
| $MB$ | `mb` | Compustat | 3-1e |
| $STOCK\_TURN$ | `stock_turn` | CRSP `dsf` | **2.2** |
| $TOTAL\_VOL$ | `total_vol` | CRSP `dsf` | **2.2** |
| $STOCK\_RET$ | `stock_ret` | CRSP `dsf` | **2.2** |
| $BETA$ | `beta` | CRSP `dport6`/`dport8` | **2.2** |

Dropped for want of data: $COMPACCT$, $CASHFLOW\_VOL$, $EARNINGS\_VOL$, $SALES\_VOL$ (five-year Compustat volatilities), $IDOSY\_VOL$, $NEG\_SKEW$ (weekly firm-specific returns), $HHI$, and $STRATEGY$.

Three of our four new variables are also *measured at a different frequency* from the paper's, because that is what `Smirk_FYear.sas` computes. Be honest about this in your own work rather than quietly relabelling:

- $STOCK\_TURN$ in the paper is average **monthly** share turnover; ours is the average **daily** `vol/shrout`.
- $TOTAL\_VOL$ in the paper is the standard deviation of **weekly** returns; ours is the standard deviation of **daily** returns.
- $BETA$ in the paper is estimated from a CAPM regression on daily returns; ours is CRSP's own `betav` from the portfolio assignment files, averaged over the year.

**How the paper estimates it.** Table 2 reports two specifications, and we build both in 2.6:

1. **OLS with year fixed effects**, standard errors clustered by firm *and* year (Petersen, 2009; Gow et al., 2010; Thompson, 2011). Adjusted $R^2 = 0.156$, $n = 17{,}057$.
2. **Firm fixed effects**, to absorb unobserved time-invariant firm characteristics. Adjusted $R^2 = 0.287$.

For reference when you look at your own output, the mean of $IV\_SKEW$ in their sample is 0.042, and the controls we keep load as follows in their column (1)/(2): $ATM\_IV$ +, $FIRM\_SIZE$ −, $LEVERAGE$ +, $MB$ +, $STOCK\_TURN$ +, $TOTAL\_VOL$ +, $BETA$ + (insignificant under OLS).

#### 2.2. Merging the CRSP control variables

The window is the same 12 months as before — 8 months before the fiscal year end through 3 months after — so the month-index trick from Step 6 of the Module 3 exercise carries over directly.

**Why we do not simply download `crsp.dsf`.** The daily stock file from 1995 is roughly 30 million rows. Instead we aggregate on the server to **permno-month sufficient statistics** — counts, sums, sums of squares, and the sum of $\ln(1+ret)$ — and reassemble the firm-year figures from those. Each one is exactly recoverable:

$$
Return = \exp\!\Big(\textstyle\sum_d \ln(1+ret_d)\Big) - 1,
\qquad
Ret\_STD = \sqrt{\frac{\sum_d ret_d^2 - (\sum_d ret_d)^2/n}{n-1}},
\qquad
Turnover = \frac{\sum_d turn_d}{n_{turn}}
$$

Sums and counts are additive across months, so this gives the *same number* as computing over the raw daily rows — the same lesson as Step 6 of the Module 3 exercise, now applied to a standard deviation and a compounded return as well as a mean.

In [ ]:
import numpy as np
import pandas as pd
import pyfixest as pf
import wrds

In [ ]:
# The Module 3 exercise wrote this file. gvkey must be read as a string --
# Compustat gvkeys carry leading zeros ("001045") that int parsing destroys.
smirk = pd.read_csv(
    "../data/Smirk_FirmYear.csv",
    dtype={"gvkey": str},
    parse_dates=["datadate"],
)
print(f"{len(smirk):,} firm-years from the Module 3 exercise")
smirk.head()

In [ ]:
db = wrds.Connection(wrds_username="leonardl")   # <-- your own WRDS username

##### 2.2.1. Daily CRSP statistics, aggregated on the server

`GROUP BY 1, 2` refers to the first two items in the `SELECT` list — a PostgreSQL convenience that saves repeating the month expression.

In [ ]:
dsf_sql = """
    SELECT permno,
           EXTRACT(YEAR FROM date)::int * 12 + EXTRACT(MONTH FROM date)::int AS month,
           COUNT(ret)                                        AS n_ret,
           SUM(ret)                                          AS sum_ret,
           SUM(ret * ret)                                    AS sum_ret2,
           SUM(CASE WHEN ret > -1 THEN LN(1 + ret) END)      AS sum_logret,
           SUM(CASE WHEN shrout > 0 THEN vol / shrout END)   AS sum_turn,
           COUNT(CASE WHEN shrout > 0 THEN vol / shrout END) AS n_turn
    FROM crsp.dsf
    WHERE ret IS NOT NULL
      AND date >= '1995-01-01'
    GROUP BY 1, 2
"""

if DSF_CACHE.exists():
    dsf_monthly = pd.read_parquet(DSF_CACHE)
else:
    dsf_monthly = db.raw_sql(dsf_sql)            # a few minutes: ~30m rows scanned
    # WRDS returns every numeric column as float64; permno is an identifier.
    dsf_monthly["permno"] = dsf_monthly["permno"].astype("int32")
    dsf_monthly["month"] = dsf_monthly["month"].astype("int32")
    dsf_monthly.to_parquet(DSF_CACHE, index=False)

print(f"{len(dsf_monthly):,} permno-months")
dsf_monthly.head()

##### 2.2.2. Reassembling the firm-year figures

Identical in shape to Step 6 of the Module 3 exercise: expand each firm-year into the 12 month keys inside its window, join on `(permno, month)`, then re-aggregate.

In [ ]:
MONTHS_BEFORE, MONTHS_AFTER = 8, 3


def month_index(s: pd.Series) -> pd.Series:
    """Months since year 0 -- differencing this reproduces SAS's intck('month', ...)."""
    return s.dt.year * 12 + s.dt.month


firm_years = smirk[["gvkey", "permno", "datadate"]].drop_duplicates().copy()
firm_years["fy_month"] = month_index(firm_years["datadate"])

keys = pd.concat(
    [firm_years.assign(month=firm_years["fy_month"] + k)
     for k in range(-MONTHS_BEFORE, MONTHS_AFTER + 1)],
    ignore_index=True,
)

totals = (
    keys.merge(dsf_monthly, on=["permno", "month"], how="inner")
        .groupby(["gvkey", "permno", "datadate"], as_index=False)
        [["n_ret", "sum_ret", "sum_ret2", "sum_logret", "sum_turn", "n_turn"]]
        .sum()
)
print(f"{len(totals):,} firm-years matched to CRSP daily data")
totals.head()

In [ ]:
# Rebuild the three SAS variables from the sufficient statistics
totals["stock_ret"] = np.expm1(totals["sum_logret"])          # exp(sum(log(1+ret))) - 1

variance = (
    (totals["sum_ret2"] - totals["sum_ret"] ** 2 / totals["n_ret"])
    / (totals["n_ret"] - 1)                                   # sample sd, like SAS std()
)
totals["total_vol"] = np.sqrt(variance.clip(lower=0))         # guard tiny negative rounding

totals["stock_turn"] = totals["sum_turn"] / totals["n_turn"]

crsp_controls = totals[["gvkey", "permno", "datadate",
                        "stock_ret", "total_vol", "stock_turn", "n_ret"]]
crsp_controls.describe()

##### 2.2.3. Market beta

In [ ]:
# What portfolio-assignment files does your CRSP subscription expose?
[t for t in db.list_tables(library="crsp") if t.startswith("dport")]

In [ ]:
db.describe_table(library="crsp", table="dport6")

In [ ]:
beta = db.raw_sql("""
    SELECT permno, year AS cyear, AVG(betav) AS beta
    FROM (
        SELECT permno, year, date, betav FROM crsp.dport6 WHERE date >= '1995-01-01'
        UNION
        SELECT permno, year, date, betav FROM crsp.dport8 WHERE date >= '1995-01-01'
    ) AS u
    GROUP BY permno, year
""")

beta["permno"] = beta["permno"].astype("int32")
beta["cyear"] = beta["cyear"].astype("int16")

print(f"{len(beta):,} permno-years")
beta.head()

##### 2.2.4. Assemble and save

In [ ]:
panel = smirk.merge(crsp_controls, on=["gvkey", "permno", "datadate"], how="left")

panel["cyear"] = panel["datadate"].dt.year.astype("int16")
panel = panel.merge(beta, on=["permno", "cyear"], how="left")

print(f"{len(panel):,} firm-years")
panel[["iv_skew", "atm_iv", "firm_size", "leverage", "mb",
       "stock_ret", "total_vol", "stock_turn", "beta"]].notna().mean().rename("non-missing")

In [ ]:
panel.to_csv("../data/Smirk_FYear.csv", index=False)
db.close()

#### 2.3. Building the estimation sample

In [ ]:
panel = pd.read_csv(
    ROOT / "data" / "Smirk_FYear.csv", dtype={"gvkey": str}, parse_dates=["datadate"]
)

DEPVAR = "iv_skew"
CONTROLS = ["atm_iv", "firm_size", "leverage", "mb",
            "stock_turn", "beta", "total_vol", "stock_ret"]

In [ ]:
reg = panel.query("nob >= 60 and at > 0").copy()
print(f"after filters: {len(reg):>7,}")

reg = reg.dropna(subset=[DEPVAR, *CONTROLS, "permno", "fyear"])
print(f"after dropping NaN: {len(reg):>7,}")

reg["fyear"] = reg["fyear"].astype(int)
reg["permno"] = reg["permno"].astype(int)

In [ ]:
def winsorize(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    """Clip a series at its own percentiles (Stata's `winsor2, cuts(1 99)`)."""
    lo, hi = s.quantile([lower, upper])
    return s.clip(lo, hi)


reg[CONTROLS] = reg[CONTROLS].apply(winsorize)
reg[[DEPVAR, *CONTROLS]].describe().T

Compare the `iv_skew` mean against the 0.042 reported by Kim et al. and the row count against their $n = 17{,}057$. They will not agree exactly — our sample filters are looser and three controls are measured at a different frequency — but an order-of-magnitude disagreement means something upstream is wrong, and this is the cheapest place to notice it.

#### 2.4. A first regression with `pyfixest`

[`pyfixest`](https://github.com/py-econometrics/pyfixest) is a Python port of R's `fixest`, and it is the closest thing in the Python ecosystem to Stata's `reghdfe`. One function, `pf.feols()`, covers everything this section needs: ordinary OLS, absorbed fixed effects, and clustered standard errors, with the degrees-of-freedom bookkeeping handled for you.

```bash
uv add pyfixest
```

The model is written as an **R-style formula string**:

```
depvar ~ x1 + x2 + x3 | fe1 + fe2
```

`~` separates the dependent variable from the regressors, `+` adds a regressor, and everything after the `|` is a **fixed effect to absorb** (we get to that in 2.6 — leave the `|` off and you get plain OLS with an intercept). Other useful pieces of the syntax: `C(fyear)` for a categorical, `x1:x2` for an interaction alone, `x1*x2` for both main effects and their interaction, `np.log(x)` for a transformation, and `i(group, x)` for a full set of interacted dummies (the workhorse of a difference-in-differences or event-study design).

The `vcov` argument chooses the standard errors. `"iid"` is the textbook assumption — independent, identically distributed residuals — and it is the wrong one here, which is the point of 2.5.

In [ ]:
fml = f"{DEPVAR} ~ " + " + ".join(CONTROLS)
print(fml)

ols = pf.feols(fml, data=reg, vcov="iid")
ols.summary()

`.summary()` prints the table; `.tidy()` returns the same numbers as a `DataFrame` you can index into, which is what we use below. `.coefplot()` draws the coefficients with their confidence intervals.

The coefficients here are already the ones we want. The standard errors are not: `"iid"` assumes the residuals are independent. In a firm-year panel they are not — a firm's smirk is correlated with its own smirk in adjacent years, and every firm's smirk moves together when the whole market gets nervous.

#### 2.5. Standard errors: clustering by firm and year

Panel residuals are correlated in two directions at once:

- **Within firm, across years** (a *time-series* dependence): firm characteristics we have not modelled persist.
- **Within year, across firms** (a *cross-sectional* dependence): common shocks — 2008 is the obvious one — hit every firm together.

Clustering on only one dimension leaves the other uncorrected. Petersen (2009) showed that in accounting and finance panels this routinely overstates *t*-statistics by a factor of two or more, and two-way clustering (Cameron, Gelbach and Miller, 2011; Thompson, 2011) is now the default in the literature. The estimator combines three one-way covariance matrices:

$$V_{firm \,\&\, year} = V_{firm} + V_{year} - V_{firm \cap year}$$

In `pyfixest` this is the `vcov` argument: `{"CRV1": "permno + fyear"}`, where CRV1 is the standard cluster-robust variance estimator and the two variables after the colon are the clustering dimensions.

In [ ]:
CLUSTER = {"CRV1": "permno + fyear"}      # two-way clustering: firm and year

ols_cl = pf.feols(fml, data=reg, vcov=CLUSTER)
ols_cl.summary()

In [ ]:
iid, clustered = ols.tidy(), ols_cl.tidy()

pd.DataFrame({
    "coef": iid["Estimate"],
    "se (iid)": iid["Std. Error"],
    "se (firm & year)": clustered["Std. Error"],
    "inflation": clustered["Std. Error"] / iid["Std. Error"],
    "t (iid)": iid["t value"],
    "t (firm & year)": clustered["t value"],
}).round(4)

The coefficients are untouched — clustering changes only the covariance matrix — but the standard errors usually grow, often substantially: factors of two or more are routine in firm-year panels, and a regressor that looked decisive under i.i.d. errors can become insignificant. This is the single most common reason a result fails to replicate.

Two things `pyfixest` is doing for you here, both worth knowing about:

- We have around 18 years, so the *year* dimension has very few clusters. Cameron and Miller (2015) recommend judging significance against the $t$ distribution with $\min(G_{firm}, G_{year}) - 1$ degrees of freedom, and that is `pyfixest`'s default (`pf.ssc(G_df="min")`).
- The two-way estimator is **not guaranteed positive semi-definite**. With very few clusters in one dimension you can get a negative variance and a `NaN` standard error. That is the estimator telling you it has run out of information, not a bug to be worked around.

#### 2.6. Fixed effects

Kim et al.'s columns (1) and (2) add year fixed effects and then firm fixed effects. Year effects absorb anything common to all firms in a given year — the level of market volatility, say. Firm effects absorb every *time-invariant* firm characteristic, observed or not: industry, exchange listing, whatever we forgot. With firm effects in the model, the coefficients are identified purely from **within-firm variation over time**, which is a much more demanding test than a cross-sectional comparison.

You could write `+ C(permno)` in the formula and let the regression build a dummy for every firm. With a couple of thousand firms that means inverting a matrix with a couple of thousand extra columns: slow, memory-hungry, and numerically unpleasant. The alternative rests on the **Frisch-Waugh-Lovell theorem** — regressing $y$ on $X$ *and* a set of dummies gives exactly the same slopes as regressing *demeaned* $y$ on *demeaned* $X$. With one fixed effect, demeaning is a `groupby().transform("mean")`; with two or more non-nested effects there is no closed form and you iterate. That iteration is what `pyfixest` runs when you put a variable after the `|`, and it never builds the dummy matrix at all.

Two details it handles that are easy to get wrong by hand:

- **Degrees of freedom.** The absorbed fixed effects are estimated parameters and have to be counted as such in the finite-sample correction. Miss this and your standard errors are too small (`pf.ssc(k_fixef="nonnested")`).
- **Singletons.** A firm observed in only one year contributes nothing to a firm fixed-effects regression, but leaving it in inflates the reported $R^2$ and distorts the degrees of freedom. `pyfixest` drops singletons by default; `fixef_rm="none"` keeps them.

In [ ]:
# Column (1): year fixed effects
year_fe = pf.feols(f"{fml} | fyear", data=reg, vcov=CLUSTER)
year_fe.summary()

In [ ]:
# Column (2): firm and year fixed effects.
# Write the fixed effects as `permno+fyear` without spaces -- pyfixest splits the
# term on spaces when it labels the rows of a regression table (2.7).
firm_fe = pf.feols(f"{fml} | permno+fyear", data=reg, vcov=CLUSTER)
firm_fe.summary()

Two things to read off the output. The **within $R^2$** rises sharply once firm effects are absorbed, mirroring the paper's 0.156 → 0.287 — most of the cross-sectional variation in the smirk is a firm-level constant. And any coefficient that *changes* between the two specifications was partly picking up cross-firm differences rather than the within-firm relationship the paper is after.

Watch the observation count too: it falls between the two models by the number of firms observed only once.

#### 2.7. Assembling a regression table

Nobody reads three `summary()` blocks. Journal tables put the specifications side by side, coefficients with standard errors beneath them in parentheses, stars for the conventional significance levels, and a block at the bottom recording which fixed effects were included.

`pf.etable()` does all of that from a list of fitted models. `labels` renames variables to the paper's notation, `felabels` names the fixed-effect rows, and `file_name="table2.tex"` writes it straight to LaTeX.

In [ ]:
LABELS = {
    "iv_skew": "IV_SKEW", "atm_iv": "ATM_IV", "firm_size": "FIRM_SIZE",
    "leverage": "LEVERAGE", "mb": "MB", "stock_turn": "STOCK_TURN",
    "beta": "BETA", "total_vol": "TOTAL_VOL", "stock_ret": "STOCK_RET",
}

pf.etable(
    [ols_cl, year_fe, firm_fe],
    labels=LABELS,
    felabels={"fyear": "Year fixed effects", "permno": "Firm fixed effects"},
    model_heads=["Pooled OLS", "Year FE", "Firm + Year FE"],
    custom_model_stats={
        "R2 (within)": [None,
                        f"{year_fe._r2_within:.3f}",
                        f"{firm_fe._r2_within:.3f}"],
    },
    notes="Standard errors clustered by firm and year, in parentheses. "
          "*** p<0.01, ** p<0.05, * p<0.10.",
)